# Econ 5200 PS2 — The Underpowered Portfolio

**ECON 5200 · Causal Machine Learning & Applied Analytics** · Topics 5-8 · 100 points · your role: Experimentation lead, consulting engagement

This is the **starter**: the structure of the set in rubric order. Fill the empty cells; keep every heading so the grader finds each step. The brief on Canvas has the full text of every step, the numbers you must hit and the rubric. Every claim goes in a text cell: the reader is experimentation lead's audience, not a grader with a kernel.

**AI is not allowed in Phases 1–3.** Generative AI is **prohibited** in Phases 1 through 3. AI is authorised and required in Phase 4. You must be able to defend every line under questioning.

## Setup

In [40]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)     # one generator, so every draw reruns the same way
pd.set_option('display.precision', 3)

import os
os.makedirs('src', exist_ok=True)   # the .py module the brief asks for goes here

## Phase 1: Reconstruct the Portfolio (30 pts)

### Step 1.1: Simulate the experiment programme (15 pts)

In [41]:
n_experiments = 200
share_succ = 0.25
succ_mean = 0.65
succ_sd = 0.2
normal_mean = 0
normal_sd = 0.04
tests_sd = 20
zero_cutoff = 0.1
rev_min, rev_max = 1000, 100000

is_real = rng.random(size=n_experiments) < share_succ

succ_draw = rng.normal(succ_mean, succ_sd, size=n_experiments)
norm_draw = rng.normal(normal_mean, normal_sd, size=n_experiments)
true_effects = np.where(is_real, succ_draw, norm_draw)

n_tested = np.exp(rng.uniform(np.log(rev_min), np.log(rev_max), size=n_experiments))
n_tested = n_tested.round().astype(int)

se = tests_sd * np.sqrt(2 / n_tested)
observed = true_effects + rng.normal(0, se)
z = observed / se
p_val = 2 * stats.norm.sf(np.abs(z))

test_dict = {'test': np.arange(1, n_experiments + 1),
             'is_real': is_real,
             'true_effects': true_effects,
             'n_tested': n_tested,
             'se': se,
             'observed': observed,
             'z': z,
             'p_val': p_val,}
df_tests = pd.DataFrame(test_dict)

print(df_tests)

print("share truly near zero:", (np.abs(df_tests['true_effects']) < zero_cutoff).mean().round(4))

     test  is_real  true_effects  n_tested     se  observed      z      p_val
0       1    False         0.030      9537  0.290    -0.332 -1.145  2.521e-01
1       2    False         0.009     93558  0.092     0.056  0.610  5.416e-01
2       3    False         0.021      8510  0.307     0.141  0.460  6.452e-01
3       4    False        -0.028     90330  0.094    -0.196 -2.086  3.696e-02
4       5     True         0.718      6655  0.347     0.675  1.947  5.156e-02
..    ...      ...           ...       ...    ...       ...    ...        ...
195   196     True         0.661     57645  0.118     0.495  4.206  2.604e-05
196   197    False         0.073     11814  0.260     0.159  0.611  5.412e-01
197   198    False        -0.066     17522  0.214    -0.131 -0.612  5.407e-01
198   199     True         0.790      3519  0.477     0.560  1.175  2.400e-01
199   200    False        -0.017      1262  0.796     0.676  0.849  3.957e-01

[200 rows x 8 columns]
share truly near zero: 0.75


*The Experiment*

The company that we are looking at is a food delivery company that runs around 200 A/B tests every year on app changes, such as the locations of buttons, colouring and more. In one example, the control group sees the regular interface, with a "manage order" section and a list menu to add items to an order that has already been placed, while the treatment group gets a large red button with the text `add items free of delivery` underneath it.

We are measuring revenue per order, and the effect is the difference in revenue per order between the treatment and control groups. Only 25% of the ideas tested have a real effect, drawn from `Normal(mean $0.65, SD $0.20)`, while the other 75% have essentially no effect, drawn from `Normal(mean $0, SD $0.04)`. An effect counts as essentially zero when `|effect| < $0.10`, and 75% of the 200 true effects fall within that cutoff. Lastly, the outcome has an SD of $20 per order, and each test has between 1,000 and 100,000 customers per group, drawn on a log-uniform scale.

### Step 1.2: Apply the client's rule (15 pts)

In [47]:
df_tests['shipped'] = (df_tests['p_val'] < 0.05) & (df_tests['observed'] > 0)

n_shipped = df_tests['shipped'].sum()
share_shipped = df_tests['shipped'].mean()
print('shipped:', n_shipped, 'of', len(df_tests), f"({share_shipped:.1%})")

df_shipped = df_tests[df_tests['shipped']]

fake_ship = np.abs(df_shipped['true_effects'] < zero_cutoff).mean()
print("shipped features that really don't have an effect:", f"{fake_ship:.1%}")

mean_true = df_shipped['true_effects'].mean()
mean_obs = df_shipped['observed'].mean()

print(f"mean true: {mean_true:.3f}   mean observed: {mean_obs:.3f}")
print(f"observed / true: {mean_obs / mean_true:.2f}x")


shipped: 28 of 200 (14.0%)
shipped features that really don't have an effect: 7.1%
mean true: 0.583   mean observed: 0.730
observed / true: 1.25x


*Client's rule*
We will be now labeling which tests actually ended up being shipped. The condition here is that the outcome was positive and also that p-value < 0.05. I added positive as we dont want to generate less revenue, that seems sub-optimal for the case of our business. From these tests we ship 14% of features that is very close to the clients desired 15%, that 14% is 28 shipped features. We also shipped 2 features out of the 28 that don't actually have a real effect making up 7.1% of the shipped features. The true mean is also $0.58 which makes the estimate 1.25x too high. This means we have the Winner's Curse where the client books a $0.78 increase despite the real effect being only $0.58.

## Phase 2: The Winner's Curse (35 pts)

### Step 2.1: Quantify the exaggeration (20 pts)

*Your answer here.*

### Step 2.2: The bootstrap where the formula fails (15 pts)

*Your answer here.*

## Phase 3: A Rule They Can Adopt (25 pts)

### Step 3.1: Design the alternative (20 pts)

*Your answer here.*

### Step 3.2: Ship the module (5 pts, its own rubric row)

The module the brief asks for. Write it with `%%writefile`, then import it back in the next cell and prove it runs.

In [43]:
%%writefile src/experiment_portfolio.py
"""Experiment-portfolio simulator and decision rules (ECON 5200 PS2)."""
from __future__ import annotations


def simulate_portfolio(n: int, seed: int, **kwargs):
    """Return the experiment log with the true effect attached."""
    ...


def apply_rule(log, rule):
    """Ship decisions under `rule`: the p < 0.05 threshold and at least one other."""
    ...


def evaluate(log, rule):
    """Realised true return, ship count and false-ship rate under `rule`."""
    ...


if __name__ == "__main__":
    ...   # demonstration plus at least three assertions that must hold

Overwriting src/experiment_portfolio.py


In [44]:
import os, sys
sys.path.insert(0, os.path.abspath('.'))
from src import experiment_portfolio as ep
# prove it runs: call the three functions on a small portfolio

## Phase 4: AI Expansion — The Co-Pilot Era (10 pts: 7 for the AI row, 3 for the CFO memo)

**Status: AI usage AUTHORISED and REQUIRED.** Use the **P.R.I.M.E. framework**.

### Task 4.1: The sequential design

*Your answer here.*

### Task 4.2: The CFO memo

*Your answer here.*

### AI Transcript

Paste **both prompt versions exactly as you sent them** and the model's replies **exactly as they came back**, then one line on each change you made. Name the model and the date. For 5200 the transcript lives in `ai-appendix.md` in your repository; link it here.

*Model and date:*

*Version 1 prompt:*

*Reply:*

*Version 2 prompt:*

*Reply:*

## Truth and headline numbers

Required, and it must stay the **last code cell**. Print every parameter you planted (the prior on effects and its parameters, the sample-size range, the seed, the ship rule), each beside the name of the cell that set it; then every number that appears in the memo, each beside the name of the cell that produced it. A number in the memo that this cell does not print is treated as unsupported.

In [45]:
print('Truth and headline numbers')
# print('prior on effects:', ..., '  set in Step 1.1')
# print('sample-size range:', ..., '  set in Step 1.1')
# print('seed:', 42, '  set in Setup')
# print('ship rule:', 'p < 0.05', '  Step 1.2')
# ... one line per memo number, with the cell that produced it

Truth and headline numbers


## Before you submit

- [ ] Every heading above is still here, in order, with work under it.
- [ ] Every number you report is produced by a cell above it, and every claim has a text cell.
- [ ] The verification checkpoint(s) are met before anything built on them.
- [ ] Phase 4: the AI transcript is the exchange, not a summary; both prompt versions are in it.
- [ ] Upload the `.ipynb`, your `ai-appendix.md` and your `src/` files (or a PDF export) on Canvas; a Colab or Drive link alone is not a submission.
- [ ] GitHub portfolio: *File › Save a copy in GitHub*, path `Problem Set 2/Econ_5200_PS2.ipynb`, with a short `README.md` in that folder.
- [ ] Run *Runtime › Restart and run all* once, top to bottom, before you hand in.